In [1]:
import os
import pandas as pd

print("Current working directory:")
print(os.getcwd())

data_path = os.path.join("..", "data", "mimic-demo", "icu")
print("Data path:", data_path)
print("Files in folder:", os.listdir(data_path)[:10])

chartevents = pd.read_csv(os.path.join(data_path, "chartevents.csv.gz"))
d_items = pd.read_csv(os.path.join(data_path, "d_items.csv.gz"))
icustays = pd.read_csv(os.path.join(data_path, "icustays.csv.gz"))

print("Data loaded successfully")

chartevents.head()

Current working directory:
c:\Users\klobn\OneDrive\Desktop\ATLAS\notebooks
Data path: ..\data\mimic-demo\icu
Files in folder: ['caregiver.csv.gz', 'chartevents.csv.gz', 'datetimeevents.csv.gz', 'd_items.csv.gz', 'icustays.csv.gz', 'ingredientevents.csv.gz', 'inputevents.csv.gz', 'outputevents.csv.gz', 'procedureevents.csv.gz']
Data loaded successfully


,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:45:00,225054,On,NaN,NaN,0.0
1,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:43:00,223769,100,100.0,%,0.0
2,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:47:00,223956,Atrial demand,NaN,NaN,0.0
3,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:47:00,224866,Yes,NaN,NaN,0.0
4,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:45:00,227341,No,0.0,NaN,0.0


In [2]:
# دمج chartevents مع d_items للحصول على أسماء القياسات
chart = chartevents.merge(d_items, on="itemid", how="left")

print("Merged successfully")
print("Number of rows:", len(chart))

# عرض الأعمدة المهمة
chart[["stay_id", "itemid", "label", "charttime", "valuenum"]].head(10)

Merged successfully
Number of rows: 668862


,stay_id,itemid,label,charttime,valuenum
0,32604416,225054,Anti Embolic Device Status,2132-12-16 00:00:00,NaN
1,32604416,223769,O2 Saturation Pulseoxymetry Alarm - High,2132-12-16 00:00:00,100.0
2,32604416,223956,Temporary Pacemaker Mode,2132-12-16 00:00:00,NaN
3,32604416,224866,Temporary Atrial Capture,2132-12-16 00:00:00,NaN
4,32604416,227341,History of falling (within 3 mnths),2132-12-16 00:00:00,0.0
5,32604416,224751,Temporary Pacemaker Rate,2132-12-16 00:00:00,52.0
6,32604416,227969,Safety Measures,2132-12-16 00:00:00,NaN
7,32604416,223935,PostTib. Pulses R,2132-12-16 00:00:00,NaN
8,32604416,223782,Pain Type,2132-12-16 00:00:00,NaN
9,32604416,224773,LLE Temp,2132-12-16 00:00:00,NaN


In [3]:
# نشوف كل أسماء القياسات (labels) الفريدة
labels = chart["label"].dropna().unique()

print("Number of unique labels:", len(labels))

# نعرض أول 100 label
labels[:100]

Number of unique labels: 1290


array(['Anti Embolic Device Status',
       'O2 Saturation Pulseoxymetry Alarm - High',
       'Temporary Pacemaker Mode', 'Temporary Atrial Capture',
       'History of falling (within 3 mnths)', 'Temporary Pacemaker Rate',
       'Safety Measures', 'PostTib. Pulses R', 'Pain Type', 'LLE Temp',
       'Pain Cause', 'Heart Rate Alarm - Low',
       'Central Venous Pressure  Alarm - Low', 'LLE Color',
       'Temporary Atrial Sens Threshold mV', 'Heart Rhythm', 'Position',
       'Pulmonary Artery Pressure Alarm - Low', 'Dorsal PedPulse L',
       'Pulmonary Artery Pressure diastolic', 'Resp Alarm - High',
       'Temporary Pacemaker Type', 'Secondary diagnosis',
       'Anti Embolic Device', 'Central Venous Pressure Alarm - High',
       'Side Rails', 'Activity', 'Temporary Venticular Stim Threshold mA',
       'PostTib. Pulses L', 'Temporary Pacemaker Wire Condition',
       'Commands', 'RLE Temp', 'Heart rate Alarm - High',
       'Respiratory Rate', 'Gait/Transferring',
       'Pulm

In [4]:
vitals = chart[
    chart["label"].isin([
        "Heart Rate",
        "Respiratory Rate",
        "Arterial Blood Pressure systolic",
        "O2 saturation pulseoxymetry"
    ])
].copy()

print("Vitals filtered:", len(vitals))
vitals[["stay_id", "label", "charttime", "valuenum"]].head()

Vitals filtered: 46891


,stay_id,label,charttime,valuenum
37,32604416,Respiratory Rate,2132-12-16 00:00:00,19.0
69,32604416,Heart Rate,2132-12-16 00:00:00,80.0
92,32604416,O2 saturation pulseoxymetry,2132-12-16 00:00:00,95.0
96,32604416,Arterial Blood Pressure systolic,2132-12-16 00:00:00,117.0
102,32604416,Heart Rate,2132-12-16 01:00:00,82.0


In [5]:
# نختار مريض واحد (stay_id)
stay_id = vitals["stay_id"].dropna().unique()[0]

print("Selected stay_id:", stay_id)

patient_data = vitals[vitals["stay_id"] == stay_id].copy()

# ترتيب البيانات حسب الوقت
patient_data["charttime"] = pd.to_datetime(patient_data["charttime"])
patient_data = patient_data.sort_values("charttime")

patient_data.head(20)

Selected stay_id: 32604416


,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
2392,10005817,20626031,32604416,12929.0,2132-12-15 14:12:00,2132-12-15 14:30:00,220210,15,15.0,insp/min,0.0,Respiratory Rate,RR,chartevents,Respiratory,insp/min,Numeric,NaN,NaN
2383,10005817,20626031,32604416,12929.0,2132-12-15 14:12:00,2132-12-15 14:30:00,220045,80,80.0,bpm,0.0,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
2389,10005817,20626031,32604416,12929.0,2132-12-15 14:12:00,2132-12-15 14:30:00,220277,100,100.0,%,0.0,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN
2395,10005817,20626031,32604416,12929.0,2132-12-15 14:12:00,2132-12-15 14:30:00,220050,126,126.0,mmHg,0.0,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0
2442,10005817,20626031,32604416,12929.0,2132-12-15 14:15:00,2132-12-15 14:30:00,220277,100,100.0,%,0.0,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN
2468,10005817,20626031,32604416,12929.0,2132-12-15 14:15:00,2132-12-15 14:30:00,220050,140,140.0,mmHg,0.0,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0
2487,10005817,20626031,32604416,12929.0,2132-12-15 14:15:00,2132-12-15 14:30:00,220045,78,78.0,bpm,0.0,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
2490,10005817,20626031,32604416,12929.0,2132-12-15 14:15:00,2132-12-15 14:30:00,220210,17,17.0,insp/min,0.0,Respiratory Rate,RR,chartevents,Respiratory,insp/min,Numeric,NaN,NaN
2763,10005817,20626031,32604416,12929.0,2132-12-15 14:30:00,2132-12-15 14:30:00,220277,100,100.0,%,0.0,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN
2807,10005817,20626031,32604416,12929.0,2132-12-15 14:30:00,2132-12-15 14:30:00,220045,79,79.0,bpm,0.0,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN


In [6]:
patient_data["label"].value_counts()

label
Respiratory Rate                    74
Heart Rate                          74
O2 saturation pulseoxymetry         74
Arterial Blood Pressure systolic    46
Name: count, dtype: int64

In [7]:
# نحدد آخر وقت (reference time)
ref_time = patient_data["charttime"].max()

print("Reference time:", ref_time)

# نحدد بداية النافذة (6 ساعات قبل)
window_start = ref_time - pd.Timedelta(hours=6)

print("Window start:", window_start)

# نفلتر البيانات داخل النافذة
window_data = patient_data[
    (patient_data["charttime"] >= window_start) &
    (patient_data["charttime"] <= ref_time)
].copy()

print("Window size:", len(window_data))

window_data.head(20)

Reference time: 2132-12-17 17:00:00
Window start: 2132-12-17 11:00:00
Window size: 21


,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
1242,10005817,20626031,32604416,20310.0,2132-12-17 11:00:00,2132-12-17 11:49:00,220277,98,98.0,%,0.0,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN
1234,10005817,20626031,32604416,20310.0,2132-12-17 11:00:00,2132-12-17 11:49:00,220210,15,15.0,insp/min,0.0,Respiratory Rate,RR,chartevents,Respiratory,insp/min,Numeric,NaN,NaN
1240,10005817,20626031,32604416,20310.0,2132-12-17 11:00:00,2132-12-17 11:49:00,220045,62,62.0,bpm,0.0,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
1291,10005817,20626031,32604416,20310.0,2132-12-17 12:00:00,2132-12-17 12:43:00,220210,12,12.0,insp/min,0.0,Respiratory Rate,RR,chartevents,Respiratory,insp/min,Numeric,NaN,NaN
1297,10005817,20626031,32604416,20310.0,2132-12-17 12:00:00,2132-12-17 12:43:00,220045,67,67.0,bpm,0.0,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
1329,10005817,20626031,32604416,20310.0,2132-12-17 12:00:00,2132-12-17 12:43:00,220277,97,97.0,%,0.0,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN
1444,10005817,20626031,32604416,20310.0,2132-12-17 13:00:00,2132-12-17 13:45:00,220277,98,98.0,%,0.0,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN
1447,10005817,20626031,32604416,20310.0,2132-12-17 13:00:00,2132-12-17 13:45:00,220045,70,70.0,bpm,0.0,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
1451,10005817,20626031,32604416,20310.0,2132-12-17 13:00:00,2132-12-17 13:45:00,220210,21,21.0,insp/min,0.0,Respiratory Rate,RR,chartevents,Respiratory,insp/min,Numeric,NaN,NaN
1455,10005817,20626031,32604416,20310.0,2132-12-17 14:00:00,2132-12-17 14:16:00,220277,97,97.0,%,0.0,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN


In [8]:
window_data["label"].value_counts()

label
O2 saturation pulseoxymetry    7
Respiratory Rate               7
Heart Rate                     7
Name: count, dtype: int64

In [9]:
# نحول البيانات إلى شكل أعمدة (pivot)
pivot_data = window_data.pivot_table(
    index="charttime",
    columns="label",
    values="valuenum",
    aggfunc="mean"
)

pivot_data.head(20)

label,Heart Rate,O2 saturation pulseoxymetry,Respiratory Rate
charttime,,,
2132-12-17 11:00:00,62.0,98.0,15.0
2132-12-17 12:00:00,67.0,97.0,12.0
2132-12-17 13:00:00,70.0,98.0,21.0
2132-12-17 14:00:00,67.0,97.0,16.0
2132-12-17 15:00:00,73.0,96.0,22.0
2132-12-17 16:00:00,76.0,95.0,22.0
2132-12-17 17:00:00,82.0,94.0,25.0


In [10]:
# آخر قيمة (current state)
current = pivot_data.iloc[-1]

print("Current values:")
print(current)

Current values:
label
Heart Rate                     82.0
O2 saturation pulseoxymetry    94.0
Respiratory Rate               25.0
Name: 2132-12-17 17:00:00, dtype: float64


In [11]:
# أول قيمة (بداية النافذة)
first = pivot_data.iloc[0]

# نحسب الفرق (trend)
trend = current - first

print("Trend over 6 hours:")
print(trend)

Trend over 6 hours:
label
Heart Rate                     20.0
O2 saturation pulseoxymetry    -4.0
Respiratory Rate               10.0
dtype: float64


In [12]:
risk = "STABLE"

if trend["Respiratory Rate"] >= 8:
    risk = "HIGH RISK"

elif trend["Heart Rate"] >= 15:
    risk = "MODERATE RISK"

elif trend["O2 saturation pulseoxymetry"] <= -3:
    risk = "WARNING"

print("Predicted Risk:", risk)

Predicted Risk: HIGH RISK


In [13]:
attention_tier = "Stable"
reason = []

# Current state
if current["Respiratory Rate"] >= 24:
    reason.append("High current respiratory rate")

if current["O2 saturation pulseoxymetry"] <= 94:
    reason.append("Borderline low oxygen saturation")

# Trend state
if trend["Heart Rate"] >= 15:
    reason.append("Heart rate rising over 6 hours")

if trend["Respiratory Rate"] >= 8:
    reason.append("Respiratory rate rising over 6 hours")

if trend["O2 saturation pulseoxymetry"] <= -3:
    reason.append("Oxygen saturation falling over 6 hours")

# Tier logic
if current["Respiratory Rate"] >= 24 and current["O2 saturation pulseoxymetry"] <= 94:
    attention_tier = "Critical Now"
elif len(reason) >= 2:
    attention_tier = "Watch Closely"
elif len(reason) == 1:
    attention_tier = "Deceptively Stable"

print("Attention Tier:", attention_tier)
print("Reasons:")
for r in reason:
    print("-", r)

Attention Tier: Critical Now
Reasons:
- High current respiratory rate
- Borderline low oxygen saturation
- Heart rate rising over 6 hours
- Respiratory rate rising over 6 hours
- Oxygen saturation falling over 6 hours


In [14]:
if attention_tier == "Stable" and len(reason) >= 2:
    attention_tier = "Deceptively Unstable"

In [15]:
# ---------------------------------
# Hidden Risk Logic (Deceptive Stability)
# ---------------------------------

hidden_risk_reasons = []

# current values are not clearly critical
current_not_critical = (
    current["Respiratory Rate"] < 24 and
    current["O2 saturation pulseoxymetry"] > 94 and
    current["Heart Rate"] < 100
)

# trend deterioration signals
if trend["Heart Rate"] >= 10:
    hidden_risk_reasons.append("Heart rate rising despite non-critical current value")

if trend["Respiratory Rate"] >= 5:
    hidden_risk_reasons.append("Respiratory rate rising despite non-critical current value")

if trend["O2 saturation pulseoxymetry"] <= -2:
    hidden_risk_reasons.append("Oxygen saturation falling despite acceptable current value")

# final hidden-risk classification
hidden_attention_tier = "No Hidden Risk"

if current_not_critical and len(hidden_risk_reasons) >= 2:
    hidden_attention_tier = "Deceptively Stable"

print("Hidden Risk Tier:", hidden_attention_tier)
print("Hidden Risk Reasons:")
for r in hidden_risk_reasons:
    print("-", r)

Hidden Risk Tier: No Hidden Risk
Hidden Risk Reasons:
- Heart rate rising despite non-critical current value
- Respiratory rate rising despite non-critical current value
- Oxygen saturation falling despite acceptable current value


In [16]:
# ===== SBP + Shock Index =====
SBP_current = current.get("Arterial Blood Pressure systolic", None)
SBP_trend = trend.get("Arterial Blood Pressure systolic", None)

if SBP_current is not None and SBP_current != 0:
    shock_index = current["Heart Rate"] / SBP_current
else:
    shock_index = None


# ===== REASONS (Main risk) =====
reason = []

if trend["Heart Rate"] >= 10:
    reason.append("Heart rate rising over 6 hours")

if trend["Respiratory Rate"] >= 8:
    reason.append("Respiratory rate rising over 6 hours")

if trend["O2 saturation pulseoxymetry"] <= -3:
    reason.append("Oxygen saturation falling over 6 hours")

# NEW 🔥
if shock_index is not None and shock_index >= 0.9:
    reason.append("High shock index")


# ===== EARLY HIDDEN RISK =====
early_hidden_reasons = []

if trend["Heart Rate"] >= 10:
    early_hidden_reasons.append("Rising heart rate")

if trend["Respiratory Rate"] >= 5:
    early_hidden_reasons.append("Rising respiratory rate")

if trend["O2 saturation pulseoxymetry"] <= -2:
    early_hidden_reasons.append("Falling oxygen saturation")

# NEW 🔥
if shock_index is not None and shock_index >= 0.8:
    early_hidden_reasons.append("Rising shock index")


# ===== TIER LOGIC =====
attention_tier = "Stable"

if current["Respiratory Rate"] >= 24:
    attention_tier = "Critical Now"
elif len(reason) >= 2:
    attention_tier = "Watch Closely"
elif len(reason) == 1:
    attention_tier = "Deceptively Stable"


# ===== HIDDEN TIER =====
early_hidden_tier = "No Early Hidden Risk"

if len(early_hidden_reasons) >= 2:
    early_hidden_tier = "Deceptively Stable"


# ===== FEATURE ROW =====
feature_row = {
    "HR_current": current["Heart Rate"],
    "RR_current": current["Respiratory Rate"],
    "SpO2_current": current["O2 saturation pulseoxymetry"],

    "HR_trend": trend["Heart Rate"],
    "RR_trend": trend["Respiratory Rate"],
    "SpO2_trend": trend["O2 saturation pulseoxymetry"],

    "SBP_current": SBP_current,
    "SBP_trend": SBP_trend,
    "shock_index": shock_index,

    "num_reasons": len(reason),
    "num_hidden_reasons": len(early_hidden_reasons),

    "attention_tier": attention_tier,
    "hidden_tier": early_hidden_tier
}

In [17]:
# ---------------------------------
# Early Hidden Risk Logic (more sensitive)
# ---------------------------------

early_hidden_reasons = []

current_not_severe = (
    current["Respiratory Rate"] < 26 and
    current["O2 saturation pulseoxymetry"] >= 94 and
    current["Heart Rate"] < 110
)

if trend["Heart Rate"] >= 10:
    early_hidden_reasons.append("Rising heart rate")

if trend["Respiratory Rate"] >= 5:
    early_hidden_reasons.append("Rising respiratory rate")

if trend["O2 saturation pulseoxymetry"] <= -2:
    early_hidden_reasons.append("Falling oxygen saturation")

early_hidden_tier = "No Early Hidden Risk"

if current_not_severe and len(early_hidden_reasons) >= 2:
    early_hidden_tier = "Deceptively Stable"

print("Early Hidden Tier:", early_hidden_tier)
print("Reasons:")
for r in early_hidden_reasons:
    print("-", r)

Early Hidden Tier: Deceptively Stable
Reasons:
- Rising heart rate
- Rising respiratory rate
- Falling oxygen saturation


In [18]:
# أول 5 مرضى (stay_ids)
stay_ids = vitals["stay_id"].dropna().unique()[:200]

print("Selected stay_ids:")
print(stay_ids)

Selected stay_ids:
[32604416 36084484 34629895 32145159 34617352 32506122 39804682 39711498
 35258379 37057036 37323533 31248398 39268883 35024147 37093652 30913302
 30864406 35065627 38507547 34600477 33177122 36059427 39864867 31338022
 35146796 34592300 38875437 37049133 38587181 32496174 38430513 35436337
 35544374 39623478 38329661 35479615 33281088 35629889 30425410 39880770
 36893762 39061571 37648963 37067082 36558922 35446858 39544395 39084876
 31494479 38229329 37200209 35679826 38540883 38017367 37293400 30932571
 39492446 35889503 32155744 35396193 30757476 34578020 35009126 36107367
 36871784 31316840 32166508 38137964 31205490 39142259 30458995 32128372
 30101877 32314488 37267577 33558396 33846653 38559363 34324099 34499716
 35026312 35636875 33083787 36753294 31552399 31959184 32554129 36091287
 32119961 37127068 32359580 30585761 39635619 33348260 32895909 38907302
 33683112 35128235 30876334 36496303 32391858 30665396 32283063 39543480
 35727289 36427705 39553978 3521

In [19]:
all_features = []

for sid in stay_ids:
    
    # فلترة مريض واحد
    patient_data = vitals[vitals["stay_id"] == sid].copy()
    
    # مهم جدًا: تحويل الوقت إلى datetime داخل اللوب
    patient_data["charttime"] = pd.to_datetime(patient_data["charttime"])

    # تأكد فيه بيانات كافية
    if len(patient_data) < 10:
        continue

    # آخر وقت
    ref_time = patient_data["charttime"].max()

    # نافذة 6 ساعات
    window_start = ref_time - pd.Timedelta(hours=6)

    window_data = patient_data[
        (patient_data["charttime"] >= window_start) &
        (patient_data["charttime"] <= ref_time)
    ].copy()

    # pivot
    pivot_data = window_data.pivot_table(
        index="charttime",
        columns="label",
        values="valuenum",
        aggfunc="mean"
    ).sort_index()

    # تأكد فيه بيانات كفاية
    if len(pivot_data) < 2:
        continue

    # current & first
    current = pivot_data.iloc[-1]
    first = pivot_data.iloc[0]
    trend = current - first

    # -----------------------
    # REASONING
    # -----------------------
    reason = []

    if current.get("Respiratory Rate", 0) >= 24:
        reason.append("High RR")

    if current.get("O2 saturation pulseoxymetry", 100) <= 94:
        reason.append("Low SpO2")

    if trend.get("Heart Rate", 0) >= 15:
        reason.append("HR rising")

    if trend.get("Respiratory Rate", 0) >= 8:
        reason.append("RR rising")

    if trend.get("O2 saturation pulseoxymetry", 0) <= -3:
        reason.append("SpO2 falling")

    # -----------------------
    # TIER
    # -----------------------
    attention_tier = "Stable"

    if current.get("Respiratory Rate", 0) >= 24 and current.get("O2 saturation pulseoxymetry", 100) <= 94:
        attention_tier = "Critical Now"
    elif len(reason) >= 2:
        attention_tier = "Watch Closely"
    elif len(reason) == 1:
        attention_tier = "Deceptively Stable"

    # -----------------------
    # FEATURE ROW
    # -----------------------
    feature_row = {
        "stay_id": sid,
        "HR_current": current.get("Heart Rate", None),
        "RR_current": current.get("Respiratory Rate", None),
        "SpO2_current": current.get("O2 saturation pulseoxymetry", None),

        "HR_trend": trend.get("Heart Rate", None),
        "RR_trend": trend.get("Respiratory Rate", None),
        "SpO2_trend": trend.get("O2 saturation pulseoxymetry", None),

        "num_reasons": len(reason),
        "attention_tier": attention_tier
    }

    all_features.append(feature_row)

features_df = pd.DataFrame(all_features)

features_df

,stay_id,HR_current,RR_current,SpO2_current,HR_trend,RR_trend,SpO2_trend,num_reasons,attention_tier
0,32604416,82.0,25.0,94.0,20.0,10.0,-4.0,5,Critical Now
1,36084484,90.0,14.0,95.0,16.0,-5.0,-1.0,1,Deceptively Stable
2,34629895,84.0,20.0,94.0,-13.0,-5.0,-2.0,1,Deceptively Stable
3,32145159,96.0,21.0,93.0,8.0,-2.0,-1.0,1,Deceptively Stable
4,34617352,36.0,0.0,NaN,-66.0,-18.0,NaN,0,Stable
...,...,...,...,...,...,...,...,...,...
133,31077365,NaN,29.0,NaN,NaN,-4.0,NaN,1,Deceptively Stable
134,35044342,NaN,30.0,NaN,NaN,1.0,NaN,1,Deceptively Stable
135,35475449,60.0,14.0,92.0,-5.0,-1.0,1.0,1,Deceptively Stable
136,34577403,85.0,19.0,99.0,1.0,-3.0,2.0,0,Stable


In [20]:
# نظرة سريعة على النتائج
print("Number of rows:", len(features_df))
print("\nAttention tier counts:")
print(features_df["attention_tier"].value_counts())

print("\nMissing values per column:")
print(features_df.isna().sum())

Number of rows: 138

Attention tier counts:
attention_tier
Stable                72
Deceptively Stable    44
Watch Closely         15
Critical Now           7
Name: count, dtype: int64

Missing values per column:
stay_id            0
HR_current        11
RR_current         7
SpO2_current      22
HR_trend          14
RR_trend           9
SpO2_trend        28
num_reasons        0
attention_tier     0
dtype: int64


In [21]:
# فلترة الصفوف المنطقية فقط
clean_features_df = features_df[
    (features_df["HR_current"].notna()) &
    (features_df["RR_current"].notna()) &
    (features_df["SpO2_current"].notna()) &
    (features_df["RR_current"] > 0)
].copy()

print("Rows before cleaning:", len(features_df))
print("Rows after cleaning:", len(clean_features_df))

clean_features_df

Rows before cleaning: 138
Rows after cleaning: 108


,stay_id,HR_current,RR_current,SpO2_current,HR_trend,RR_trend,SpO2_trend,num_reasons,attention_tier
0,32604416,82.0,25.0,94.0,20.0,10.0,-4.0,5,Critical Now
1,36084484,90.0,14.0,95.0,16.0,-5.0,-1.0,1,Deceptively Stable
2,34629895,84.0,20.0,94.0,-13.0,-5.0,-2.0,1,Deceptively Stable
3,32145159,96.0,21.0,93.0,8.0,-2.0,-1.0,1,Deceptively Stable
5,32506122,62.0,13.0,97.0,-9.0,-4.0,-2.0,0,Stable
...,...,...,...,...,...,...,...,...,...
131,30849778,82.0,26.0,94.0,-5.0,1.0,-4.0,3,Critical Now
132,38333427,88.0,26.0,96.0,0.0,0.0,-2.0,1,Deceptively Stable
135,35475449,60.0,14.0,92.0,-5.0,-1.0,1.0,1,Deceptively Stable
136,34577403,85.0,19.0,99.0,1.0,-3.0,2.0,0,Stable


In [22]:
print("Final dataset size:", len(clean_features_df))

print("\nAttention tier distribution:")
print(clean_features_df["attention_tier"].value_counts())

Final dataset size: 108

Attention tier distribution:
attention_tier
Stable                54
Deceptively Stable    34
Watch Closely         13
Critical Now           7
Name: count, dtype: int64


In [23]:
def build_feature_row_for_stay(sid, vitals):
    # فلترة مريض واحد
    patient_data = vitals[vitals["stay_id"] == sid].copy()
    patient_data["charttime"] = pd.to_datetime(patient_data["charttime"])

    # تأكد فيه بيانات كافية
    if len(patient_data) < 10:
        return None

    # آخر وقت
    ref_time = patient_data["charttime"].max()

    # نافذة 6 ساعات
    window_start = ref_time - pd.Timedelta(hours=6)

    window_data = patient_data[
        (patient_data["charttime"] >= window_start) &
        (patient_data["charttime"] <= ref_time)
    ].copy()

    # pivot
    pivot_data = window_data.pivot_table(
        index="charttime",
        columns="label",
        values="valuenum",
        aggfunc="mean"
    ).sort_index()

    # تأكد فيه بيانات كفاية
    if len(pivot_data) < 2:
        return None

    # current & first
    current = pivot_data.iloc[-1]
    first = pivot_data.iloc[0]
    trend = current - first

    # reasoning
    reason = []

    if current.get("Respiratory Rate", 0) >= 24:
        reason.append("High RR")

    if current.get("O2 saturation pulseoxymetry", 100) <= 94:
        reason.append("Low SpO2")

    if trend.get("Heart Rate", 0) >= 15:
        reason.append("HR rising")

    if trend.get("Respiratory Rate", 0) >= 8:
        reason.append("RR rising")

    if trend.get("O2 saturation pulseoxymetry", 0) <= -3:
        reason.append("SpO2 falling")

    # tier
    attention_tier = "Stable"

    if current.get("Respiratory Rate", 0) >= 24 and current.get("O2 saturation pulseoxymetry", 100) <= 94:
        attention_tier = "Critical Now"
    elif len(reason) >= 2:
        attention_tier = "Watch Closely"
    elif len(reason) == 1:
        attention_tier = "Deceptively Stable"

    # feature row
    feature_row = {
        "stay_id": sid,
        "HR_current": current.get("Heart Rate", None),
        "RR_current": current.get("Respiratory Rate", None),
        "SpO2_current": current.get("O2 saturation pulseoxymetry", None),

        "HR_trend": trend.get("Heart Rate", None),
        "RR_trend": trend.get("Respiratory Rate", None),
        "SpO2_trend": trend.get("O2 saturation pulseoxymetry", None),

        "num_reasons": len(reason),
        "attention_tier": attention_tier
    }

    return feature_row

In [24]:
all_features = []

for sid in stay_ids:
    row = build_feature_row_for_stay(sid, vitals)
    if row is not None:
        all_features.append(row)

    print("Rows collected before cleaning:", len(all_features))

features_df = pd.DataFrame(all_features)

clean_features_df = features_df[
    (features_df["HR_current"].notna()) &
    (features_df["RR_current"].notna()) &
    (features_df["SpO2_current"].notna()) &
    (features_df["RR_current"] > 0)
].copy()

print("Rows after cleaning:", len(clean_features_df))

clean_features_df.head()

Rows collected before cleaning: 1
Rows collected before cleaning: 2
Rows collected before cleaning: 3
Rows collected before cleaning: 4
Rows collected before cleaning: 5
Rows collected before cleaning: 6
Rows collected before cleaning: 7
Rows collected before cleaning: 8
Rows collected before cleaning: 9
Rows collected before cleaning: 10
Rows collected before cleaning: 11
Rows collected before cleaning: 12
Rows collected before cleaning: 13
Rows collected before cleaning: 14
Rows collected before cleaning: 15
Rows collected before cleaning: 16
Rows collected before cleaning: 17
Rows collected before cleaning: 18
Rows collected before cleaning: 19
Rows collected before cleaning: 20
Rows collected before cleaning: 21
Rows collected before cleaning: 22
Rows collected before cleaning: 23
Rows collected before cleaning: 24
Rows collected before cleaning: 25
Rows collected before cleaning: 26
Rows collected before cleaning: 27
Rows collected before cleaning: 28
Rows collected before cleanin

,stay_id,HR_current,RR_current,SpO2_current,HR_trend,RR_trend,SpO2_trend,num_reasons,attention_tier
0,32604416,82.0,25.0,94.0,20.0,10.0,-4.0,5,Critical Now
1,36084484,90.0,14.0,95.0,16.0,-5.0,-1.0,1,Deceptively Stable
2,34629895,84.0,20.0,94.0,-13.0,-5.0,-2.0,1,Deceptively Stable
3,32145159,96.0,21.0,93.0,8.0,-2.0,-1.0,1,Deceptively Stable
5,32506122,62.0,13.0,97.0,-9.0,-4.0,-2.0,0,Stable


In [25]:
print("Final dataset size:", len(clean_features_df))

print("\nAttention tier distribution:")
print(clean_features_df["attention_tier"].value_counts())

Final dataset size: 108

Attention tier distribution:
attention_tier
Stable                54
Deceptively Stable    34
Watch Closely         13
Critical Now           7
Name: count, dtype: int64


In [26]:
print("Final dataset size:", len(clean_features_df))
print(clean_features_df["attention_tier"].value_counts())

Final dataset size: 108
attention_tier
Stable                54
Deceptively Stable    34
Watch Closely         13
Critical Now           7
Name: count, dtype: int64


In [27]:
# نشوف كم صف فيه SBP داخل vitals
bp_rows = vitals[vitals["label"] == "Arterial Blood Pressure systolic"].copy()

print("Number of SBP rows:", len(bp_rows))
bp_rows.head()

Number of SBP rows: 5525


,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
96,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-16 00:02:00,220050,117,117.0,mmHg,0.0,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0
114,10005817,20626031,32604416,6770.0,2132-12-16 01:00:00,2132-12-16 01:04:00,220050,125,125.0,mmHg,0.0,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0
120,10005817,20626031,32604416,6770.0,2132-12-16 02:00:00,2132-12-16 02:11:00,220050,110,110.0,mmHg,0.0,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0
132,10005817,20626031,32604416,6770.0,2132-12-16 03:00:00,2132-12-16 03:00:00,220050,111,111.0,mmHg,0.0,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0
171,10005817,20626031,32604416,6770.0,2132-12-16 04:00:00,2132-12-16 04:21:00,220050,127,127.0,mmHg,0.0,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0


In [28]:
# كم stay_id عنده SBP؟
bp_stays = bp_rows["stay_id"].dropna().nunique()
all_stays = vitals["stay_id"].dropna().nunique()

print("Stay IDs with SBP:", bp_stays)
print("All stay IDs in vitals:", all_stays)

Stay IDs with SBP: 65
All stay IDs in vitals: 140


In [29]:
features_df.head()

,stay_id,HR_current,RR_current,SpO2_current,HR_trend,RR_trend,SpO2_trend,num_reasons,attention_tier
0,32604416,82.0,25.0,94.0,20.0,10.0,-4.0,5,Critical Now
1,36084484,90.0,14.0,95.0,16.0,-5.0,-1.0,1,Deceptively Stable
2,34629895,84.0,20.0,94.0,-13.0,-5.0,-2.0,1,Deceptively Stable
3,32145159,96.0,21.0,93.0,8.0,-2.0,-1.0,1,Deceptively Stable
4,34617352,36.0,0.0,NaN,-66.0,-18.0,NaN,0,Stable
